In [25]:
!pip install langchain langchain-community sentence-transformers faiss-cpu transformers accelerate


In [31]:
! pip install pypdf
! pip install unstructured

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.8/167.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 12.8 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=0181894ec01c87a8420f44db236bd43cdcad23d2a2766e854f8318166c644a21
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


#Load your documents

In [32]:
import os

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    UnstructuredHTMLLoader,
)

docs_dir = "./docs"
documents = []

for file in os.listdir(docs_dir):
    file_path = os.path.join(docs_dir, file)
    ext = file.lower().split(".")[-1]

    try:
        if ext == "md":
            loader = TextLoader(file_path, encoding="utf-8")

        elif ext == "pdf":
            loader = PyPDFLoader(file_path)

        elif ext in ["html", "htm"]:
            loader = UnstructuredHTMLLoader(file_path)

        else:
            print(f"Skipping unsupported file: {file}")
            continue

        documents.extend(loader.load())

    except Exception as e:
        print(f"Failed to load {file}: {e}")

print("Total documents loaded:", len(documents))



Skipping unsupported file: .ipynb_checkpoints
Total documents loaded: 21


# Split text into chunks

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)
print("Chunks:", len(chunks))

Chunks: 86


# Create embeddings using HuggingFace (FREE)

In [34]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# Creating FAISS vector database

In [35]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)
print("FAISS ready")


FAISS ready


# Load a HuggingFace LLM (local chatbot)

In [36]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")

llm = HuggingFacePipeline(pipeline=pipe)


Device set to use cuda:0


#Create RAG Query function

In [39]:
def ask(question):
    docs = vectorstore.similarity_search(question, k=3)
    context = "\n\n".join(d.page_content for d in docs)

    prompt = f"Context:\n{context}\n\nQuestion:\n{question}\nAnswer:"

    return llm.invoke(prompt)

print(ask("who is deepesh lodhi"))

Context:
Team behind the Blog

We are a small team of people in NSS IIT Delhi who love to explore, read and create intriguing content. We love what we do, and we do it with passion.

Deepesh Lodhi

Secretary, NSS IIT Delhi

I am a fun loving person who loves listening music, always equipped to meet new people, make new friends, learn new things and currently exploring this world of possibilities.

LinkedIn

Mail

Navya Tripathi

Secretary, NSS IIT Delhi

Faculty Co-Ordinator

member

Prof Ankesh

Faculty Co-Ordinator NSS IITD

email

General Secretaries

member

Nidhi Pandey

General Secretary

insta

email

member

Sneha Bhargava

General Secretary

linkedin

insta

email

Secretaries

member

Rishabh Chirania

Secretary

linkedin

insta

email

member

Vaishali Rathod

Secretary

linkedin

insta

email

member

Harshit Jain

Secretary

linkedin

insta

email

member

Tarun Kumar

Secretary

linkedin

insta

email

member

Gaurav Meena

Prof Dibakar

Faculty Advisor (Education)

email